In [ ]:
import sys
sys.executable

In [ ]:
sys.version


In [ ]:
import torch
torch.__version__

In [ ]:
x = torch.tensor([
    [1,2,3],
    [4,5,6]
])
print(x)
print(x.shape)
print(x[1,2])

In [ ]:
print(x[0,:])
print(x[:2,:])

In [ ]:
y = x.T
print(y)

In [ ]:
scores = x @ x.T
print(scores)

In [ ]:
x = torch.tensor([
    [
        [1,2],
        [3,4],
        [5,6]
    ],
    [
        [1,2],
        [3,4],
        [5,6]
    ],
    [
        [1,2],
        [3,4],
        [5,6]
    ]
])
print(x)
print(x.size)

# soft max
$$
w_i = \frac{e^{s_i}}{\sum_j e^{s_j}}
$$

In [ ]:
x = torch.tensor([
    [1,2,3],
    [4,5,6]
], dtype=torch.float32) # 这里要声明不是整数张量

weights = x.softmax(dim = -1)
print(weights)

# 分配权重

In [ ]:
values = torch.tensor([
    [2,3],
    [3,4],
    [7,8]
],dtype=torch.float32)

context = weights @ values
print(context)



In [ ]:
# 真正的kqv 和softmax

q = torch.tensor([
    [1.0,3.0]
])

key = torch.tensor([
    [1,1],
    [2,2],
    [3,5]
],dtype=torch.float32)

scores = q @ key.T
print(scores)

# 因为长度会增加2倍方差 因此除以根号2 标准化方差
d_k = key.shape[-1]
scaled_scores = scores/(d_k ** 0.5)
attention = scaled_scores.softmax(dim = -1)


print(attention)
print(attention.sum(dim=-1))

context = attention @ values
values = torch.tensor([
    [2,3],
    [3,4],
    [7,8]
],dtype=torch.float32)


print(context)

In [ ]:
print('q=',q)
print('q @ key.T = attention')
print('key = ',key)
print('attention @ values = context')
print('values = ',values)
# 这里按照token 的顺序都是横着写下来的

In [ ]:
# 产生kqv 也在这里完成

X = torch.tensor([
    [1.0, 2.0],
    [3.0, 4.0],
    [5.0, 6.0]
])

W_Q = torch.tensor([
    [1.0, 0.0],
    [0.0, 2.0]
])

q = X@W_Q
print(q)

In [ ]:
W_K = torch.tensor([
    [0.0, 1.0],
    [1.0, 0.0]
],dtype=torch.float32)

W_V = torch.tensor([
    [1.0, 0.0],
    [0.0, 1.0]
],dtype=torch.float32)

key = X@W_K
values = X@W_V

score = q @ key.T
attention = (score/(key.shape[-1] ** 0.5)).softmax(dim = -1)
context = attention @ values

print(context)

In [ ]:
from a1_tensor_basic import attention_forward


torch.manual_seed(0)

X = torch.randn(6, 16)
W_Q = torch.randn(16, 16)
W_K = torch.randn(16, 16)
W_V = torch.randn(16, 16)

context = attention_forward(X,W_Q,W_K,W_V)

# 这里开始训练embedding

In [ ]:
embedding = torch.nn.Embedding(1000,64)
# here torch generate a random 1000 vocab embedding with N(0,1) distribution
token_ids = torch.tensor([0, 1, 0, 1, 3, 3])
token_vectors = embedding(token_ids)
print(X.shape)
#加入位置信息
position_embedding = torch.nn.Embedding(128,64)
position_ids = torch.arange(len(token_ids))
postition_vectors = position_embedding(position_ids)

X = token_vectors + postition_vectors
print(X)

In [ ]:
# initialized as trainable parameter
W_Q = torch.nn.Parameter(torch.randn(64, 64) * (64 ** -0.5))
W_K = torch.nn.Parameter(torch.randn(64, 64) * (64 ** -0.5))
W_V = torch.nn.Parameter(torch.randn(64, 64) * (64 ** -0.5))

print(W_K.requires_grad)
print(1)


In [ ]:
context = attention_forward(
    X, W_Q, W_K, W_V, prefix_len=4
)

print(context.shape)

In [ ]:
# 创建输出层：64 维 → 1000 个 token 的分数
output_head = torch.nn.Linear(64, 1000)

logits = output_head(context)

print(logits.shape)
print(logits.argmax(-1))

In [ ]:
# find loss

target = torch.tensor([0,1])
masked_logits = logits[4:]

loss = torch.nn.functional.cross_entropy(masked_logits,target=target)
print(loss.item())

loss.backward()

print(W_Q.grad.shape)
print(W_Q.grad.norm())

# 现在真正的梯度更新

In [ ]:
parameters = (
    list(embedding.parameters()) +
    list(position_embedding.parameters()) +
    [W_K,W_Q,W_V]+
    list(output_head.parameters())
)

optimizer = torch.optim.SGD(parameters)


optimizer.zero_grad()
X = embedding(token_ids)+position_embedding(position_ids)
context = attention_forward(X,W_Q,W_K,W_V,prefix_len=4)
logits = output_head(context)
loss = torch.nn.functional.cross_entropy(logits[4:],target)
loss.backward()
optimizer.step()

# I am going to make it generate english

In [ ]:
chars = "abcdefghijklmnopqrstuvwxyz,. "

# 字符 → ID
char_to_id = {char: i for i, char in enumerate(chars)}

# ID → 字符
id_to_char = {i: char for char, i in char_to_id.items()}

mask_id = len(chars)       
vocab_size = len(chars) + 1  
training_set = '''Learning to Pay Attention in a World That Never Stops
Every morning, millions of people reach for their phones before they reach for a glass of water. A screen lights up, messages arrive, and the outside world enters the room before the day has properly begun. Some people check the weather. Others read headlines, answer a colleague, or watch a short video that leads to another. None of these actions seems particularly important on its own. Together, however, they establish a pattern: attention moves outward before a person has decided where it should go. By breakfast, the mind may already feel busy, even though very little has actually happened.
This experience raises a practical question. How can people remain curious, informed, and connected without allowing every available signal to determine their thoughts? The answer requires more than turning off notifications or purchasing a notebook. It involves understanding attention as something shaped by habits, environments, expectations, and choices. Attention influences what people remember, how they learn, and which problems they notice. It also affects their relationships. A conversation changes when someone listens carefully. A familiar street changes when someone walks slowly enough to observe it. The world contains more detail than any individual can process, so selection is unavoidable.
Consider a student named Maya, who begins a university course on computer programming. During the first week, she watches several tutorials and copies examples into her laptop. The programs run successfully, and she feels encouraged. Yet when an assignment asks her to solve a slightly different problem, she freezes. She recognizes the terms but cannot decide what to write. Her difficulty does not mean the tutorials were useless. They helped her become familiar with the language. What they did not provide was enough practice making decisions independently. Familiarity had arrived before understanding, and she had mistaken one for the other.
Maya changes her approach. Instead of watching an entire lesson at once, she pauses after a short example. She closes the video and tries to rebuild the program from memory. When she gets stuck, she writes down the exact point of confusion. Sometimes she cannot remember a command. Sometimes she understands the command but does not know why it belongs in that location. These are different problems, and separating them makes her questions more useful. Over time, her notebook becomes a record of decisions rather than a collection of copied answers. Progress feels slower during each session but becomes clearer across weeks.
The same distinction appears in many fields. A person learning a language may recognize hundreds of words while struggling to order lunch. A musician may understand a discussion of rhythm while losing the beat during a performance. Someone studying mathematics may follow a proof line by line without being able to explain why its central idea works. Recognition matters, but it is only one part of learning. Practical understanding grows when people retrieve information, apply it, notice errors, and revise their approach. The uncomfortable moment of attempting something without immediate guidance often reveals what further practice needs to address.
Modern technology can support this process in remarkable ways. A learner can access lectures, simulations, dictionaries, and specialized software from a single device. Questions that once required a visit to a library can sometimes be explored within minutes. People separated by oceans can work on the same document, compare observations, and discuss an unfinished idea. These possibilities expand access to knowledge. Their value, however, depends partly on how they are used. A powerful tool can help someone investigate a question deeply, or it can make it easier to avoid the difficult thinking that the question requires.
Imagine two students using an artificial intelligence assistant to prepare for an examination. The first asks for complete solutions and reads them quickly. The second attempts each problem, identifies a doubtful step, and asks for feedback on that step. Both students receive useful information, but their activities differ. One mainly evaluates whether an explanation looks convincing. The other practices forming and checking a chain of reasoning. Neither pattern guarantees a particular result, and circumstances matter. Still, the comparison shows why the presence of technology tells us less than the behavior surrounding it. Tools participate in learning routines; they do not define those routines alone.
Good routines also depend on the physical environment. A desk covered with unrelated papers creates different possibilities from a desk prepared for one specific task. A phone placed beside a textbook remains available even when its owner intends to ignore it. An open browser with twenty tabs offers many exits from a difficult paragraph. Small changes can make a chosen activity easier to begin and easier to continue. Before studying, Maya now opens the files she needs, puts her phone in a drawer, and writes a single sentence describing the session’s purpose. The preparation takes only a few minutes.
Her purpose is deliberately concrete. “Study programming” leaves too many decisions unresolved. “Write a function that counts repeated words and test it on three sentences” gives her a visible target. A narrow task does not eliminate difficulty, but it helps her distinguish productive struggle from aimless movement. If the function fails, she can examine the input, the expected output, and the actual result. If it works, she can explain why and choose a next step. Clear targets create useful stopping points as well. Without them, a person may continue working long after concentration has become too weak to support careful thought.
Rest belongs in this discussion because attention changes throughout the day. People are not equally alert at every hour, and effort does not produce identical results under all conditions. A difficult task may become manageable after a walk, a meal, or a night of sleep. This does not make persistence unimportant. It makes observation important. Learning to recognize the difference between ordinary resistance and genuine exhaustion helps people use their time more sensibly. Maya begins recording when she studies and how the session goes. She discovers that her best work often happens in the morning, before messages and meetings accumulate.
Outside the classroom, attention shapes the quality of everyday relationships. During a conversation, it is possible to hear another person’s words while preparing a response, checking a screen, or remembering an unfinished obligation. Listening carefully requires leaving some mental space for what has not yet been said. A friend describing a disappointing experience may want advice, but may first need to explain why the event mattered. A quick solution can miss that meaning. Asking one thoughtful question sometimes contributes more than offering five suggestions. The value comes from responding to the actual person and situation rather than to a familiar pattern.
Attention also influences how people encounter their communities. Someone who travels the same route every day may stop noticing the shops, trees, and public spaces along it. Then a small disruption changes the journey. A delayed bus leads to a walk. A closed road requires a different turn. Suddenly, the neighborhood becomes visible again. There is a notice about a local concert, a bench that needs repair, or a new business occupying an empty storefront. Such observations rarely feel dramatic, yet they form part of civic awareness. People can respond to local needs only when those needs become noticeable to them.
This is one reason curiosity benefits from both direction and freedom. Directed curiosity helps a person investigate a defined question: Why does this program fail? How does this machine work? What explains a change in a neighborhood? Open curiosity allows unexpected details to become questions in the first place. A balanced learning life makes room for both. Constantly pursuing a fixed goal can narrow perception, while endless exploration can prevent useful conclusions. The challenge is to move between exploration and commitment deliberately. Sometimes the right action is to follow a surprising connection. Sometimes it is to finish testing the original idea.
Keeping records can make that movement easier. A short journal, research log, or working notebook preserves observations that memory might otherwise reshape. The most useful entries need not be elegant. A date, a question, an attempted method, and a result may be enough. For Maya, a failed program becomes less frustrating when she can compare it with earlier attempts. She can see which error has disappeared and which remains. Records also protect against exaggerated impressions. A difficult afternoon does not necessarily mean a month has gone badly. Looking back at concrete work provides a more reliable basis for judging progress.
The desire to measure progress, however, introduces another challenge. Some measures are easy to collect but incomplete. Hours spent at a desk do not reveal how much a person understood. Pages read do not show which ideas can be used later. A long sequence of completed exercises may reflect either growing skill or repeated practice with tasks that have become comfortable. Measurements help when they match the purpose of the activity. If Maya wants to become better at solving unfamiliar problems, she needs occasional unfamiliar problems. A record of tutorial hours can provide context, but it cannot answer that question by itself.
Uncertainty is therefore a normal part of meaningful work. Before testing an idea, a person may not know whether it will succeed. Before asking a question, a student may not know whether the confusion is shared by others. Before beginning a project, a team may have only a rough estimate of the effort required. Responding well to uncertainty involves making assumptions visible and choosing actions that produce useful information. A small experiment can clarify a large disagreement. A rough draft can reveal that two collaborators imagined different audiences. An early conversation can prevent weeks of work on incompatible expectations.
Working with others also exposes the limits of individual attention. Different people notice different details because they bring different experiences, skills, and concerns. During a group project, one person may focus on whether a system works, another on whether its instructions are understandable, and another on whether people can access it easily. These perspectives can create friction, particularly when time is limited. They can also improve the final result. Productive collaboration requires explaining observations clearly enough that others can evaluate them. “This feels wrong” may begin a discussion, but a specific example gives the group something concrete to investigate.
As Maya becomes more confident, she volunteers to help a classmate. Explaining a simple program reveals gaps she had not noticed while working alone. She can produce the correct result but struggles to describe an intermediate step. Rather than hiding the uncertainty, she runs a smaller example and examines what happens. The conversation becomes a shared investigation. Teaching, in this form, is valuable because it makes understanding observable. An explanation must connect ideas in a sequence another person can follow. When that sequence breaks, the break identifies a place where both explanation and understanding can improve.
Over the semester, Maya’s habits become more stable without becoming perfectly consistent. She still loses time to distractions. Some study sessions end with little progress. Occasionally, she chooses an approach that turns out to be inefficient. What changes is her response. She is better able to identify what happened and adjust the next attempt. A mistake becomes evidence about a method or assumption rather than a final judgment about her ability. This shift does not remove disappointment, but it gives disappointment a practical destination. Instead of asking only whether she performed well, she asks what the experience allows her to do differently.
The wider world will continue to compete for people’s attention. New services, devices, and forms of communication will create useful opportunities alongside fresh distractions. No single habit will solve every difficulty, and different lives require different arrangements. Yet ordinary decisions remain consequential. Choosing a clear question, attempting an answer, listening without interruption, recording an observation, and returning after a setback all influence what becomes possible. A meaningful learning life develops through these repeated actions. It grows when people give important things enough time to become understandable and allow understanding to change what they do next.'''